# Exercise 8 - Ensemble on MNIST data

In [5]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.ensemble import VotingClassifier

In [2]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(np.uint8)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=10000, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=10000, random_state=42, stratify=y_train)
print(X_train.shape, X_val.shape, X_test.shape)

(50000, 784) (10000, 784) (10000, 784)


In [4]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
extra = ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)
svm = SVC(kernel="rbf", probability=True, random_state=42)

rf.fit(X_train, y_train)
extra.fit(X_train, y_train)
svm.fit(X_train, y_train)
models = {"Random Forest": rf, "Extra Trees": extra, "SVM": svm}
for name, model in models.items():
    print(name, accuracy_score(y_val, model.predict(X_val)))

Random Forest 0.9709
Extra Trees 0.9731
SVM 0.9798


In [6]:
voting = VotingClassifier(estimators=[("rf", rf), ("extra", extra), ("svm", svm)], voting="hard", n_jobs=-1)
voting.fit(X_train, y_train)
y_val_pred = voting.predict(X_val)

print("Voting accuracy:", accuracy_score(y_val, y_val_pred))

Voting accuracy: 0.9752


In [7]:
for name, model in models.items():
    print(name, accuracy_score(y_val, model.predict(X_val)))
print("Voting", accuracy_score(y_val, y_val_pred))

Random Forest 0.9709
Extra Trees 0.9731
SVM 0.9798
Voting 0.9752


In [8]:
soft_voting = VotingClassifier(estimators=[("rf", rf), ("extra", extra), ("svm", svm)], voting="soft", n_jobs=-1)
soft_voting.fit(X_train, y_train)
y_val_pred_soft = soft_voting.predict(X_val)

print("Soft voting accuracy:", accuracy_score(y_val, y_val_pred_soft))

Soft voting accuracy: 0.981


In [9]:
best_ensemble = voting
y_test_pred = best_ensemble.predict(X_test)
ensemble_acc = accuracy_score(y_test, y_test_pred)
print("Ensemble test accuracy:", ensemble_acc)

for name, model in models.items():
    acc = accuracy_score(y_test, model.predict(X_test))
    print(name, acc)

Ensemble test accuracy: 0.9728
Random Forest 0.9657
Extra Trees 0.9706
SVM 0.9787


In [10]:
for name, model in models.items():
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name}: {acc:.4f}")
    print(f"Improvement: {(ensemble_acc - acc) * 100:.2f} percentage points")

Random Forest: 0.9657
Improvement: 0.71 percentage points
Extra Trees: 0.9706
Improvement: 0.22 percentage points
SVM: 0.9787
Improvement: -0.59 percentage points


The important idea here is isn't simply that "three models are better than one." It's that different classifiers make different mistakes. Combining their predictions can cancel out some individual errors which is the basic motivation behind voting ensembles.